# Esquema: clasificación binaria (MVP manual)

Target **texto** → 0/1 manual. Varios clasificadores en **`make_pipeline(StandardScaler, modelo)`**.

| Modelos en pipeline | |
|---------------------|---|
| Regresión logística, KNN, árbol, bosque aleatorio, SVM | |

Siguiente: [07.b binaria](../07.b-ejemplos-supervisados/02-clasificacion-binaria.ipynb).


## 1. CSV, tipos y faltantes

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("data/datos_spam.csv")
print(df.dtypes)
print("\nFaltantes:\n", df.isna().sum())
display(df)


Palabras_En_Texto    float64
Etiqueta_Texto        object
dtype: object

Faltantes:
 Palabras_En_Texto    1
Etiqueta_Texto       1
dtype: int64


,Palabras_En_Texto,Etiqueta_Texto
0,3.0,no
1,8.0,no
2,12.0,no
3,25.0,si
4,30.0,si
5,45.0,si
6,5.0,no
7,NaN,si
8,22.0,si
9,7.0,no


## 2. Target texto → 0/1

In [5]:
MAPA_BINARIO = {"no": 0, "si": 1}
df = df.dropna(subset=["Etiqueta_Texto"]).copy()
y = df["Etiqueta_Texto"].str.strip().str.lower().map(MAPA_BINARIO).astype(int)


## 3. Feature numérica

In [6]:
palabras = df["Palabras_En_Texto"].astype(float)
X = pd.DataFrame({"Palabras_En_Texto": palabras.fillna(palabras.median())})


## 4. Split estratificado

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## 5. Varios modelos en Pipeline

In [8]:
def build_models():
    """Misma lista que 07.b (comenta entradas para excluir modelos)."""
    from sklearn.ensemble import (
        GradientBoostingClassifier,
        HistGradientBoostingClassifier,
        RandomForestClassifier,
    )
    from sklearn.linear_model import LogisticRegression, SGDClassifier
    from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.svm import SVC
    from sklearn.tree import DecisionTreeClassifier
    from xgboost import XGBClassifier
    from catboost import CatBoostClassifier

    return {
        "LogisticRegression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        "SGDClassifier": SGDClassifier(
            loss="log_loss",
            max_iter=2000,
            tol=1e-3,
            random_state=RANDOM_STATE,
        ),
        "SVC": SVC(random_state=RANDOM_STATE),
        "OneVsOneClassifier": OneVsOneClassifier(SVC(random_state=RANDOM_STATE)),
        "OneVsRestClassifier": OneVsRestClassifier(SVC(random_state=RANDOM_STATE)),
        "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "DecisionTree": DecisionTreeClassifier(
            criterion="gini",
            splitter="best",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features=None,
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            random_state=RANDOM_STATE,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            criterion="gini",
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1,
            min_weight_fraction_leaf=0.0,
            max_features="sqrt",
            max_leaf_nodes=None,
            min_impurity_decrease=0.0,
            bootstrap=True,
            oob_score=False,
            max_samples=None,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(
            random_state=RANDOM_STATE,
            verbosity=0,
            n_estimators=100,
            eval_metric="logloss",
            n_jobs=-1,
        ),
        "CatBoost": CatBoostClassifier(
            random_state=RANDOM_STATE,
            verbose=False,
            iterations=100,
            allow_writing_files=False,
        ),
    }


RANDOM_STATE = 42
MODELS = build_models()


filas = []
mejor_nombre, mejor_acc, mejor_pred = None, -1.0, None

for nombre, modelo in MODELS.items():
    pipe = make_pipeline(StandardScaler(), modelo)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, pred)
    filas.append({"modelo": nombre, "accuracy": acc})
    if acc > mejor_acc:
        mejor_acc, mejor_nombre, mejor_pred = acc, nombre, pred

display(pd.DataFrame(filas).sort_values("accuracy", ascending=False).round(4))

print(f"\nMejor accuracy: {mejor_nombre} ({mejor_acc:.2f})")
print(classification_report(
    y_test, mejor_pred, target_names=["No spam (0)", "Spam (1)"]
))


,modelo,accuracy
0,LogisticRegression,1.0
1,KNN,1.0
2,DecisionTree,1.0
3,RandomForest,1.0
4,SVC,1.0



Mejor accuracy: LogisticRegression (1.00)
              precision    recall  f1-score   support

 No spam (0)       1.00      1.00      1.00         1
    Spam (1)       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2

